In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
"""
================================================================================
PFE 2026 - PRÉTRAITEMENT CHB-MIT SUR KAGGLE
================================================================================
Auteurs: Mohamed Yacine Khatib, Yousri Thabet
Encadrant: Riheb Naili

Dataset: /kaggle/input/seizure-epilepcy-chb-mit-eeg-dataset-pediatric
================================================================================
"""

# ============================================================================
# SNIPPET 1: CONFIGURATION KAGGLE
# ============================================================================

import os
import numpy as np
import mne
import gc
import json
import re
from scipy import signal
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("📦 SNIPPET 1: CONFIGURATION CHB-MIT KAGGLE")
print("="*80)

class Config:
    """Configuration pour Kaggle - Dataset officiel CHB-MIT"""
    
    # Chemin KAGGLE - Dataset monté automatiquement
    DATA_DIR = "/kaggle/input/seizure-epilepcy-chb-mit-eeg-dataset-pediatric/chb-mit-scalp-eeg-database-1.0.0"
    OUTPUT_DIR = "/kaggle/working/CHB-MIT_14ch"
    
    # Paramètres du signal
    TARGET_SFREQ = 256      # Hz
    WINDOW_SIZE = 10        # secondes
    WINDOW_SAMPLES = TARGET_SFREQ * WINDOW_SIZE  # 2560
    
    # Paramètres de filtrage (Europe - 50 Hz)
    LOW_FREQ = 1.0
    HIGH_FREQ = 40.0
    NOTCH_FREQ = 50.0
    
    # Paramètres des crises
    PRE_ICTAL_MINUTES = 5
    PRE_ICTAL_SECONDS = PRE_ICTAL_MINUTES * 60  # 300s

os.makedirs(Config.OUTPUT_DIR, exist_ok=True)

print(f"✅ Dataset source: {Config.DATA_DIR}")
print(f"✅ Dossier sortie: {Config.OUTPUT_DIR}")
print(f"✅ Fréquence: {Config.TARGET_SFREQ} Hz")
print(f"✅ Fenêtre: {Config.WINDOW_SIZE}s ({Config.WINDOW_SAMPLES} samples)")
print(f"✅ Filtrage: {Config.LOW_FREQ}-{Config.HIGH_FREQ} Hz + notch {Config.NOTCH_FREQ} Hz")
print(f"✅ Pré-ictal: {Config.PRE_ICTAL_MINUTES} minutes")

In [ ]:
# ============================================================================
# SNIPPET 2: EXPLORATION DU DATASET KAGGLE
# ============================================================================

print("\n" + "="*80)
print("🔍 SNIPPET 2: EXPLORATION DU DATASET KAGGLE")
print("="*80)

def explore_kaggle_dataset():
    """Explore la structure du dataset Kaggle CHB-MIT"""
    
    if not os.path.exists(Config.DATA_DIR):
        print(f"❌ Dataset non trouvé: {Config.DATA_DIR}")
        print("   Vérifie que tu as bien ajouté le dataset via 'Add Data'")
        return
    
    # Lister tous les patients
    patients = [d for d in os.listdir(Config.DATA_DIR) 
                if os.path.isdir(os.path.join(Config.DATA_DIR, d)) and d.startswith('chb')]
    
    print(f"✅ Dataset trouvé!")
    print(f"📁 {len(patients)} patients: {sorted(patients)}")
    
    # Compter les fichiers par patient
    total_edf = 0
    for patient in sorted(patients):
        patient_path = os.path.join(Config.DATA_DIR, patient)
        edf_files = [f for f in os.listdir(patient_path) if f.endswith('.edf')]
        summary_file = os.path.join(patient_path, f"{patient}-summary.txt")
        has_summary = os.path.exists(summary_file)
        
        print(f"\n   📁 {patient}: {len(edf_files)} fichiers .edf")
        if has_summary:
            print(f"      ✅ {patient}-summary.txt présent")
            # Lire quelques lignes du summary
            try:
                with open(summary_file, 'r') as f:
                    lines = f.readlines()[:5]
                    for line in lines:
                        if 'Seizure' in line:
                            print(f"      📊 {line.strip()}")
            except:
                pass
        else:
            print(f"      ❌ Fichier summary manquant")
        
        total_edf += len(edf_files)
    
    print(f"\n📊 TOTAL: {total_edf} fichiers .edf, {len(patients)} patients")
    return patients

patients_list = explore_kaggle_dataset()

In [ ]:
# ============================================================================
# SNIPPET 3: DÉFINITION DU PROCESSEUR CHB-MIT KAGGLE
# ============================================================================

print("\n" + "="*80)
print("🧠 SNIPPET 3: DÉFINITION DU PROCESSEUR CHB-MIT")
print("="*80)

class CHBMITProcessor:
    """
    Processeur CHB-MIT optimisé pour Kaggle.
    
    Transforme le montage bipolaire en 14 canaux unipolaires standard.
    """
    
    CHANNELS_14 = [
        'Fp1', 'Fp2', 'F3', 'F4', 'F7', 'F8', 'Fz',
        'C3', 'C4', 'Cz',
        'T3', 'T4', 'P3', 'P4'
    ]
    
    BIPOLAR_MAP = {
        'FP1-F3': 'Fp1', 'FP2-F4': 'Fp2',
        'F3-C3': 'F3', 'F4-C4': 'F4',
        'C3-P3': 'C3', 'C4-P4': 'C4',
        'P3-O1': 'P3', 'P4-O2': 'P4',
        'FP1-F7': 'F7', 'FP2-F8': 'F8',
        'F7-T7': 'F7', 'F8-T8': 'F8',
        'T7-P7': 'T3', 'T8-P8': 'T4',
        'FZ-CZ': 'Fz', 'CZ-PZ': 'Cz',
        'P7-O1': 'P3', 'P8-O2': 'P4',
        'T7-FT9': 'T3', 'FT9-FT10': 'Fz', 'FT10-T8': 'T4'
    }
    
    @staticmethod
    def is_already_processed(edf_path: str) -> bool:
        """Vérifie si le fichier est déjà traité."""
        basename = os.path.splitext(os.path.basename(edf_path))[0]
        x_file = os.path.join(Config.OUTPUT_DIR, f"X_{basename}.npy")
        y_file = os.path.join(Config.OUTPUT_DIR, f"y_{basename}.npy")
        meta_file = os.path.join(Config.OUTPUT_DIR, f"meta_{basename}.json")
        return os.path.exists(x_file) and os.path.exists(y_file) and os.path.exists(meta_file)
    
    @staticmethod
    def fix_duplicates(raw):
        """Renomme les canaux dupliqués."""
        ch_names = raw.ch_names
        new_names = []
        counts = {}
        
        for name in ch_names:
            if name not in counts:
                counts[name] = 0
                new_names.append(name)
            else:
                counts[name] += 1
                new_names.append(f"{name}_{counts[name]}")
        
        if new_names != ch_names:
            raw.rename_channels(dict(zip(ch_names, new_names)))
        return raw
    
    @staticmethod
    def load_seizures(edf_path: str):
        """Charge les annotations de crises depuis summary.txt."""
        seizure_periods = []
        filename = os.path.basename(edf_path)
        patient_folder = os.path.dirname(edf_path)
        patient_id = os.path.basename(patient_folder)
        summary_file = os.path.join(patient_folder, f"{patient_id}-summary.txt")
        
        if not os.path.exists(summary_file):
            return seizure_periods
        
        try:
            with open(summary_file, 'r', encoding='utf-8', errors='ignore') as f:
                content = f.read()
            
            sections = content.split('File Name:')
            for section in sections:
                if filename in section:
                    start_times = re.findall(r'Seizure(?:\s\d+)?\s*Start Time:\s*(\d+)\s*seconds', section)
                    end_times = re.findall(r'Seizure(?:\s\d+)?\s*End Time:\s*(\d+)\s*seconds', section)
                    
                    for s, e in zip(start_times, end_times):
                        seizure_periods.append((float(s), float(e)))
                    break
        except Exception:
            pass
        
        return seizure_periods
    
    def process_file(self, edf_path: str, save_output: bool = True):
        """Traite un fichier CHB-MIT."""
        filename = os.path.basename(edf_path)
        basename = os.path.splitext(filename)[0]
        
        try:
            # 1. Chargement
            raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
            raw = self.fix_duplicates(raw)
            
            # 2. Rééchantillonnage
            if raw.info['sfreq'] != Config.TARGET_SFREQ:
                raw.resample(Config.TARGET_SFREQ)
            
            # 3. Scaling V → μV
            if np.abs(raw.get_data()).max() < 0.01:
                raw._data = raw._data * 1e6
            
            # 4. Filtrage
            raw.filter(Config.LOW_FREQ, Config.HIGH_FREQ, verbose=False)
            raw.notch_filter(Config.NOTCH_FREQ, verbose=False)
            
            # 5. Mapping des canaux
            channel_sources = {ch: [] for ch in self.CHANNELS_14}
            available_channels = []
            
            for idx, ch_name in enumerate(raw.ch_names):
                ch_clean = ch_name.split('_')[0]
                if ch_clean in self.BIPOLAR_MAP:
                    target = self.BIPOLAR_MAP[ch_clean]
                    if target in self.CHANNELS_14:
                        channel_sources[target].append(idx)
                        if target not in available_channels:
                            available_channels.append(target)
            
            available_channels = [ch for ch in self.CHANNELS_14 if ch in available_channels]
            
            if len(available_channels) < 13:
                return None, None, None
            
            # 6. Chargement des crises
            seizures = self.load_seizures(edf_path)
            
            # 7. Segmentation
            win_samples = Config.WINDOW_SAMPLES
            n_windows = raw.n_times // win_samples
            
            X_list = []
            timestamps = []
            
            for i in range(n_windows):
                start = i * win_samples
                end = start + win_samples
                
                window_data = np.zeros((len(available_channels), win_samples), dtype=np.float32)
                for j, ch in enumerate(available_channels):
                    idxs = channel_sources[ch]
                    if len(idxs) == 1:
                        window_data[j] = raw.get_data(picks=idxs[0], start=start, stop=end)[0]
                    elif len(idxs) > 1:
                        data = raw.get_data(picks=idxs, start=start, stop=end)
                        window_data[j] = np.mean(data, axis=0)
                
                X_list.append(window_data)
                timestamps.append(i * Config.WINDOW_SIZE)
            
            if not X_list:
                return None, None, None
            
            X = np.array(X_list, dtype=np.float32)
            
            # 8. Labeling
            labels = np.zeros(len(X), dtype=np.int32)
            for sz_start, sz_end in seizures:
                pre_start = sz_start - Config.PRE_ICTAL_SECONDS
                for i, ts in enumerate(timestamps):
                    center = ts + Config.WINDOW_SIZE/2
                    if sz_start <= center <= sz_end:
                        labels[i] = 2
                    elif pre_start <= center < sz_start:
                        labels[i] = 1
            
            # 9. Sauvegarde
            if save_output and len(X) > 0:
                np.save(os.path.join(Config.OUTPUT_DIR, f"X_{basename}.npy"), X)
                np.save(os.path.join(Config.OUTPUT_DIR, f"y_{basename}.npy"), labels)
                np.save(os.path.join(Config.OUTPUT_DIR, f"channels_{basename}.npy"), 
                       np.array(available_channels, dtype=object))
                
                metadata = {
                    'filename': basename,
                    'patient': os.path.basename(os.path.dirname(edf_path)),
                    'channels': available_channels,
                    'seizures': seizures,
                    'windows': len(X),
                    'labels': [int(np.sum(labels==0)), int(np.sum(labels==1)), int(np.sum(labels==2))],
                    'fs': Config.TARGET_SFREQ,
                    'window_sec': Config.WINDOW_SIZE,
                    'data_range': [float(X.min()), float(X.max())],
                    'data_mean': float(X.mean()),
                    'data_std': float(X.std())
                }
                
                with open(os.path.join(Config.OUTPUT_DIR, f"meta_{basename}.json"), 'w') as f:
                    json.dump(metadata, f, indent=2)
            
            return X, labels, available_channels
            
        except Exception as e:
            print(f"   ❌ Erreur {basename}: {e}")
            return None, None, None

print("✅ CHBMITProcessor défini")

In [ ]:
# ============================================================================
# SNIPPET 4: TRAITEMENT PAR LOT KAGGLE
# ============================================================================

print("\n" + "="*80)
print("🚀 SNIPPET 4: TRAITEMENT CHB-MIT SUR KAGGLE")
print("="*80)

def process_all_chbmit_kaggle(skip_existing: bool = True):
    """Traite tous les fichiers CHB-MIT du dataset Kaggle."""
    
    if not os.path.exists(Config.DATA_DIR):
        print(f"❌ Dataset non trouvé: {Config.DATA_DIR}")
        return None
    
    # Trouver tous les fichiers .edf
    edf_files = []
    for root, dirs, files in os.walk(Config.DATA_DIR):
        for file in files:
            if file.endswith('.edf'):
                edf_files.append(os.path.join(root, file))
    
    print(f"📁 {len(edf_files)} fichiers .edf trouvés")
    
    processor = CHBMITProcessor()
    already_processed = 0
    
    if skip_existing:
        for edf_path in edf_files:
            if processor.is_already_processed(edf_path):
                already_processed += 1
        print(f"   • Déjà traités: {already_processed}")
        print(f"   • À traiter: {len(edf_files) - already_processed}")
    
    results = {
        'total': len(edf_files),
        'processed': 0,
        'skipped': already_processed,
        'failed': 0,
        'windows': 0,
        'seizure_windows': 0,
        'seizure_files': 0
    }
    
    for i, edf_path in enumerate(edf_files, 1):
        filename = os.path.basename(edf_path)
        
        if skip_existing and processor.is_already_processed(edf_path):
            print(f"\n[{i}/{len(edf_files)}] {filename} - ⏩ SKIP")
            continue
        
        print(f"\n[{i}/{len(edf_files)}] {filename} - 🆕 TRAITEMENT")
        
        X, y, channels = processor.process_file(edf_path, save_output=True)
        
        if X is not None:
            results['processed'] += 1
            results['windows'] += len(X)
            results['seizure_windows'] += np.sum(y >= 1)
            if np.sum(y >= 1) > 0:
                results['seizure_files'] += 1
            print(f"   ✅ {len(X)} fenêtres, {np.sum(y>=1)} crises")
            print(f"   ✅ Canaux: {len(channels)}/14")
        else:
            results['failed'] += 1
            print(f"   ❌ Échec")
        
        gc.collect()
        
        # Rapport progression
        if i % 10 == 0 or i == len(edf_files):
            print(f"\n📊 PROGRESSION: {i}/{len(edf_files)}")
            print(f"   ✅ Nouveaux: {results['processed']}")
            print(f"   ⏩ Skippés: {results['skipped']}")
            print(f"   💾 Fenêtres: {results['windows']:,}")
    
    return results

print("✅ Fonction process_all_chbmit_kaggle() définie")

In [ ]:
# SNIPPET 5: EXÉCUTION DU TRAITEMENT
# ============================================================================

print("\n" + "="*80)
print("⚡ SNIPPET 5: EXÉCUTION DU TRAITEMENT")
print("="*80)
print("\n⚠️  DÉCOMMENTE LA LIGNE CI-DESSOUS POUR LANCER LE TRAITEMENT")
print("   Le traitement complet prend ~2-3 heures sur Kaggle")
print("   Assure-toi d'avoir activé le GPU (Settings → Accelerator)\n")

# DÉCOMMENTE POUR LANCER LE TRAITEMENT COMPLET
chb_results = process_all_chbmit_kaggle(skip_existing=True)

# Version test : traite seulement les 5 premiers fichiers
def test_kaggle_pipeline():
    """Test le pipeline sur les 5 premiers fichiers."""
    print("\n🧪 TEST: Traitement des 5 premiers fichiers...")
    
    edf_files = []
    for root, dirs, files in os.walk(Config.DATA_DIR):
        for file in files:
            if file.endswith('.edf'):
                edf_files.append(os.path.join(root, file))
        break  # Premier patient seulement
    
    processor = CHBMITProcessor()
    for edf_path in edf_files[:5]:
        filename = os.path.basename(edf_path)
        print(f"\n   📄 Test: {filename}")
        X, y, channels = processor.process_file(edf_path, save_output=True)
        if X is not None:
            print(f"      ✅ {len(X)} fenêtres")
        else:
            print(f"      ❌ Échec")

# DÉCOMMENTE POUR TESTER
test_kaggle_pipeline()

In [ ]:
# ============================================================================
# CELLULE DE VÉRIFICATION RAPIDE - TEST CHB-MIT
# ============================================================================

import os
import numpy as np
import json

print("="*80)
print("🔍 VÉRIFICATION RAPIDE - TEST CHB-MIT")
print("="*80)

output_dir = "/kaggle/working/CHB-MIT_14ch"

# 1. VÉRIFIER QUE DES FICHIERS ONT ÉTÉ CRÉÉS
print("\n📁 1. Fichiers créés ?")
if not os.path.exists(output_dir):
    print("   ❌ Aucun dossier de sortie trouvé")
    print("   → Le test n'a RIEN produit")
else:
    x_files = [f for f in os.listdir(output_dir) if f.startswith('X_') and f.endswith('.npy')]
    print(f"   ✅ {len(x_files)} fichiers X_*.npy trouvés")
    
    if len(x_files) == 0:
        print("   ❌ Aucun fichier de données généré")
    else:
        print(f"   ✅ TEST RÉUSSI - {len(x_files)} fichiers traités")

# 2. VÉRIFIER LE CONTENU D'UN FICHIER
print("\n📊 2. Contenu des fichiers ?")
if len(x_files) > 0:
    # Prendre le premier fichier
    first_file = x_files[0]
    basename = first_file[2:-4]
    
    # Charger X
    x_path = os.path.join(output_dir, first_file)
    X = np.load(x_path)
    
    # Charger y
    y_path = os.path.join(output_dir, f"y_{basename}.npy")
    y = np.load(y_path) if os.path.exists(y_path) else None
    
    # Charger channels
    ch_path = os.path.join(output_dir, f"channels_{basename}.npy")
    channels = np.load(ch_path, allow_pickle=True) if os.path.exists(ch_path) else None
    
    print(f"\n   📄 Fichier test: {basename}")
    print(f"      • Shape X: {X.shape}")
    print(f"      • Shape y: {y.shape if y is not None else '❌'}")
    print(f"      • Canaux: {len(channels) if channels is not None else '❌'}/14")
    print(f"      • Range X: [{X.min():.1f}, {X.max():.1f}] μV")
    
    if y is not None:
        unique, counts = np.unique(y, return_counts=True)
        print(f"      • Labels: ", end='')
        for u, c in zip(unique, counts):
            print(f"{u}:{c} ", end='')
        print()
    
    # ✅ CRITÈRES DE RÉUSSITE
    print("\n   📋 CRITÈRES DE RÉUSSITE:")
    
    critere1 = X.shape[1] == 14
    critere2 = X.shape[2] == 2560
    critere3 = not np.isnan(X).any()
    critere4 = np.abs(X).max() > 1.0  # échelle μV
    critere5 = y is not None and len(y) == len(X)
    
    print(f"      • 14 canaux: {'✅' if critere1 else '❌'}")
    print(f"      • 2560 samples: {'✅' if critere2 else '❌'}")
    print(f"      • Pas de NaN: {'✅' if critere3 else '❌'}")
    print(f"      • Échelle μV: {'✅' if critere4 else '⚠️'}")
    print(f"      • Labels OK: {'✅' if critere5 else '❌'}")
    
    if all([critere1, critere2, critere3, critere5]):
        print("\n   ✅✅✅ TEST RÉUSSI ! Pipeline fonctionnel ✅✅✅")
    else:
        print("\n   ⚠️⚠️⚠️ TEST PARTIEL - Vérifie les erreurs ⚠️⚠️⚠️")

# 3. VÉRIFIER LA STRUCTURE GLOBALE
print("\n📁 3. Structure du dossier de sortie:")
if os.path.exists(output_dir):
    all_files = os.listdir(output_dir)
    x_count = len([f for f in all_files if f.startswith('X_')])
    y_count = len([f for f in all_files if f.startswith('y_')])
    ch_count = len([f for f in all_files if f.startswith('channels_')])
    meta_count = len([f for f in all_files if f.endswith('.json')])
    
    print(f"   • Fichiers X_*.npy: {x_count}")
    print(f"   • Fichiers y_*.npy: {y_count}")
    print(f"   • Fichiers channels_*.npy: {ch_count}")
    print(f"   • Fichiers meta_*.json: {meta_count}")
    
    if x_count == y_count == ch_count == meta_count and x_count > 0:
        print(f"\n   ✅ Structure cohérente - {x_count} fichiers complets")
    else:
        print(f"\n   ⚠️ Structure incohérente - fichiers manquants")

print("\n" + "="*80)
print("✅ VÉRIFICATION TERMINÉE")
print("="*80)

In [ ]:
# ============================================================================
# VÉRIFICATION DES LABELS
# ============================================================================

output_dir = "/kaggle/working/CHB-MIT_14ch"

# Compte les fichiers y_
y_files = [f for f in os.listdir(output_dir) if f.startswith('y_') and f.endswith('.npy')]
print(f"📁 Fichiers y_*.npy trouvés : {len(y_files)}")

if len(y_files) > 0:
    # Test sur le premier fichier
    y = np.load(os.path.join(output_dir, y_files[0]))
    unique, counts = np.unique(y, return_counts=True)
    print(f"\n✅ LABELS TROUVÉS !")
    print(f"   • Shape: {y.shape}")
    print(f"   • Distribution: ", end='')
    for u, c in zip(unique, counts):
        print(f"{u}:{c} ", end='')
    print()
else:
    print(f"\n❌ AUCUN LABEL TROUVÉ - LE CODE NE LABELLISE PAS !")

In [ ]:
# ============================================================================
# SNIPPET : VÉRIFICATION TOTALE - 100% DE CONFIANCE
# ============================================================================
# Exécute CE SNIPPET pour vérifier ABSOLUMENT TOUT en 30 secondes
# ============================================================================

import os
import numpy as np
import json
import matplotlib.pyplot as plt
from scipy import signal

print("="*80)
print("🔬 VÉRIFICATION TOTALE - 100% DE CONFIANCE")
print("="*80)

output_dir = "/kaggle/working/CHB-MIT_14ch"

if not os.path.exists(output_dir):
    print(f"❌ Dossier non trouvé: {output_dir}")
    print("   → Exécute d'abord le traitement !")
    exit()

# ============================================================================
# 1. VÉRIFICATION QUANTITATIVE - TOUS LES FICHIERS
# ============================================================================

print("\n📊 1. VÉRIFICATION QUANTITATIVE")
print("-"*50)

x_files = sorted([f for f in os.listdir(output_dir) if f.startswith('X_') and f.endswith('.npy')])
y_files = sorted([f for f in os.listdir(output_dir) if f.startswith('y_') and f.endswith('.npy')])
ch_files = sorted([f for f in os.listdir(output_dir) if f.startswith('channels_') and f.endswith('.npy')])
meta_files = sorted([f for f in os.listdir(output_dir) if f.endswith('.json')])

print(f"📁 Fichiers trouvés:")
print(f"   • X_*.npy     : {len(x_files):3d} fichiers")
print(f"   • y_*.npy     : {len(y_files):3d} fichiers")
print(f"   • channels_*  : {len(ch_files):3d} fichiers")
print(f"   • meta_*.json : {len(meta_files):3d} fichiers")

if len(x_files) == len(y_files) == len(ch_files) == len(meta_files):
    print(f"\n✅ COHÉRENCE: {len(x_files)} fichiers complets")
else:
    print(f"\n❌ INCOHÉRENCE: fichiers manquants!")
    print(f"   X: {len(x_files)}, y: {len(y_files)}, ch: {len(ch_files)}, meta: {len(meta_files)}")

# ============================================================================
# 2. VÉRIFICATION QUALITATIVE - FICHIER SPÉCIFIQUE
# ============================================================================

print("\n📊 2. VÉRIFICATION QUALITATIVE")
print("-"*50)

# 🔴🔴🔴 CHANGE ICI LE NOM DU FICHIER À TESTER 🔴🔴🔴
TARGET_FILE = "chb13_58"  # ← Remplace par le fichier que tu veux tester
# Exemples: "chb01_03", "chb06_01", "chb13_12", etc.

print(f"🧪 Test sur fichier ciblé: {TARGET_FILE}")

# Construire les chemins
x_path = os.path.join(output_dir, f"X_{TARGET_FILE}.npy")
y_path = os.path.join(output_dir, f"y_{TARGET_FILE}.npy")
ch_path = os.path.join(output_dir, f"channels_{TARGET_FILE}.npy")
meta_path = os.path.join(output_dir, f"meta_{TARGET_FILE}.json")

# Vérifier que le fichier existe
if not os.path.exists(x_path):
    print(f"\n❌ Fichier non trouvé: {TARGET_FILE}")
    print(f"   • Vérifie le nom exact dans le dossier")
    print(f"   • Fichiers disponibles: {[f[2:-4] for f in x_files[:5]]}...")
else:
    # Chargement
    X = np.load(x_path)
    y = np.load(y_path)
    channels = np.load(ch_path, allow_pickle=True)
    
    with open(meta_path, 'r') as f:
        meta = json.load(f)
    
    # Vérifications
    print(f"\n   📐 SHAPE:")
    print(f"      • X: {X.shape} → {'✅' if X.shape[1]==14 and X.shape[2]==2560 else '❌'}")
    print(f"      • y: {y.shape} → {'✅' if len(y)==X.shape[0] else '❌'}")
    print(f"      • canaux: {len(channels)}/14 → {'✅' if len(channels)==14 else '❌'}")
    
    print(f"\n   ⚖️ ÉCHELLE:")
    print(f"      • min: {X.min():.1f} μV")
    print(f"      • max: {X.max():.1f} μV")
    print(f"      • mean: {X.mean():.1f} ± {X.std():.1f} μV")
    print(f"      → {'✅ μV' if np.abs(X).max() > 1 else '⚠️ Volts?'}")
    
    print(f"\n   🧪 NAN/INF:")
    print(f"      • NaN: {'✅ NON' if not np.isnan(X).any() else '❌ OUI'}")
    print(f"      • Inf: {'✅ NON' if not np.isinf(X).any() else '❌ OUI'}")
    
    print(f"\n   🎯 LABELS:")
    unique, counts = np.unique(y, return_counts=True)
    total = len(y)
    print(f"      • Distribution:")
    seizure_windows = 0
    for u, c in zip(unique, counts):
        name = {0: 'Interictal', 1: 'PREICTAL ⚠️', 2: 'ICTAL 🚨'}.get(u, str(u))
        print(f"         {name}: {c:4d} ({c/total*100:5.1f}%)")
        if u >= 1:
            seizure_windows += c
    
    print(f"\n      • Fenêtres de crise: {seizure_windows}")
    print(f"      • {'✅ PRÉDICTION (label 1 présent)' if 1 in unique else '❌ DÉTECTION SEULEMENT'}")
    
    print(f"\n   📋 CRISES (meta.json):")
    if 'seizures' in meta and len(meta['seizures']) > 0:
        print(f"      • {len(meta['seizures'])} crise(s) détectée(s):")
        for i, (start, end) in enumerate(meta['seizures'][:3]):
            print(f"         {i+1}. {start:.1f}s - {end:.1f}s ({end-start:.1f}s)")
    else:
        print(f"      • Aucune crise dans ce fichier")
    
    print(f"\n   ✅ BILAN: {'✅ TOUT EST OK' if seizure_windows > 0 and 1 in unique else '⚠️ VÉRIFIER'}")
# ============================================================================
# 3. STATISTIQUES GLOBALES
# ============================================================================

print("\n📊 3. STATISTIQUES GLOBALES")
print("-"*50)

total_windows = 0
total_seizure_windows = 0
files_with_seizures = 0
total_preictal = 0
total_ictal = 0

for x_file in x_files[:20]:  # Échantillon représentatif
    try:
        basename = x_file[2:-4]
        y = np.load(os.path.join(output_dir, f"y_{basename}.npy"))
        
        total_windows += len(y)
        seizure_win = np.sum(y >= 1)
        total_seizure_windows += seizure_win
        total_preictal += np.sum(y == 1)
        total_ictal += np.sum(y == 2)
        
        if seizure_win > 0:
            files_with_seizures += 1
    except:
        pass

print(f"📊 STATISTIQUES (échantillon {min(20, len(x_files))} fichiers):")
print(f"   • Total fenêtres: {total_windows:,}")
print(f"   • Fenêtres PREICTAL (⚠️): {total_preictal:,}")
print(f"   • Fenêtres ICTAL (🚨): {total_ictal:,}")
print(f"   • Fichiers avec crises: {files_with_seizures}")

# ============================================================================
# 4. VISUALISATION RAPIDE (optionnel)
# ============================================================================

print("\n📊 4. VISUALISATION RAPIDE")
print("-"*50)

try:
    if len(x_files) > 0:
        test_file = x_files[0]
        basename = test_file[2:-4]
        X = np.load(os.path.join(output_dir, f"X_{basename}.npy"))
        y = np.load(os.path.join(output_dir, f"y_{basename}.npy"))
        channels = np.load(os.path.join(output_dir, f"channels_{basename}.npy"), allow_pickle=True)
        
        fig, axes = plt.subplots(2, 2, figsize=(15, 8))
        fig.suptitle(f'Vérification finale - {basename}', fontsize=14)
        
        # 1. Exemple de signaux
        ax = axes[0, 0]
        time = np.arange(2560) / 256
        for i in range(min(5, X.shape[1])):
            ax.plot(time, X[0, i, :] + i*200, label=channels[i])
        ax.set_xlabel('Temps (s)')
        ax.set_ylabel('Amplitude (μV) + offset')
        ax.set_title('Première fenêtre - 5 canaux')
        ax.legend(loc='upper right', fontsize=8)
        ax.grid(True, alpha=0.3)
        
        # 2. Distribution des labels
        ax = axes[0, 1]
        unique, counts = np.unique(y, return_counts=True)
        colors = ['green', 'orange', 'red']
        labels = ['Interictal', 'Preictal', 'Ictal']
        bars = ax.bar(labels[:len(unique)], counts[:len(unique)], color=colors[:len(unique)])
        ax.set_title(f'Distribution des labels - Total: {len(y)} fenêtres')
        ax.set_ylabel('Nombre de fenêtres')
        for bar, count in zip(bars, counts):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                   f'{count}', ha='center', va='bottom')
        
        # 3. Spectre
        ax = axes[1, 0]
        f, Pxx = signal.welch(X[0, 0, :], fs=256)
        ax.semilogy(f, Pxx)
        ax.axvspan(1, 40, alpha=0.2, color='green', label='Bande 1-40 Hz')
        ax.axvline(50, color='red', linestyle='--', label='Notch 50 Hz')
        ax.set_xlabel('Fréquence (Hz)')
        ax.set_ylabel('Densité spectrale')
        ax.set_title('Spectre - Canal Fp1')
        ax.legend()
        ax.set_xlim(0, 100)
        ax.grid(True, alpha=0.3)
        
        # 4. Statistiques canal
        ax = axes[1, 1]
        ch_means = X.mean(axis=(0, 2))
        ch_stds = X.std(axis=(0, 2))
        x_pos = np.arange(len(channels))
        ax.bar(x_pos, ch_means, yerr=ch_stds, capsize=5, alpha=0.7, color='steelblue')
        ax.set_xlabel('Canal')
        ax.set_ylabel('Amplitude (μV)')
        ax.set_title('Moyenne ± écart-type par canal')
        ax.set_xticks(x_pos)
        ax.set_xticklabels(channels, rotation=45, ha='right')
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        print("✅ Visualisation générée avec succès")
        
except Exception as e:
    print(f"⚠️ Visualisation non disponible: {e}")

# ============================================================================
# 5. RAPPORT FINAL - 100% CONFIANCE
# ============================================================================

print("\n" + "="*80)
print("📋 RAPPORT FINAL - NIVEAU DE CONFIANCE")
print("="*80)

confidence_score = 0
max_score = 8

# Critères de confiance
if len(x_files) == len(y_files) == len(ch_files) == len(meta_files) and len(x_files) > 0:
    confidence_score += 2
    print("✅ CRITÈRE 1: Cohérence fichiers ✓")

if len(x_files) >= 13:
    confidence_score += 1
    print("✅ CRITÈRE 2: Au moins 13 fichiers traités ✓")

if 'test_file' in locals() and X.shape[1] == 14 and X.shape[2] == 2560:
    confidence_score += 1
    print("✅ CRITÈRE 3: Shape correcte (14,2560) ✓")

if 'X' in locals() and not np.isnan(X).any() and not np.isinf(X).any():
    confidence_score += 1
    print("✅ CRITÈRE 4: Pas de NaN/Inf ✓")

if 'X' in locals() and np.abs(X).max() > 1:
    confidence_score += 1
    print("✅ CRITÈRE 5: Échelle μV ✓")

if 'y' in locals() and len(np.unique(y)) == 3:
    confidence_score += 1
    print("✅ CRITÈRE 6: 3 classes (0,1,2) ✓")

if 'y' in locals() and 1 in np.unique(y):
    confidence_score += 1
    print("✅ CRITÈRE 7: PRÉDICTION (label 1 présent) ✓")

if total_preictal > 0:
    confidence_score += 1
    print("✅ CRITÈRE 8: Fenêtres préictales détectées ✓")

print("\n" + "="*80)
print(f"🎯 NIVEAU DE CONFIANCE: {confidence_score}/{max_score}")
print("="*80)

if confidence_score >= 7:
    print("""
╔═══════════════════════════════════════════════════════════════════════╗
║                                                                       ║
║   ✅✅✅ CONFIANCE TOTALE - 100% PRÊT POUR PHASE 2 ! ✅✅✅          ║
║                                                                       ║
║   • Pipeline de prétraitement:    ✅ VALIDÉ                         ║
║   • Chargement des crises:        ✅ FONCTIONNEL                    ║
║   • Labeling 3 classes:           ✅ 0,1,2                          ║
║   • PRÉDICTION (preictal):        ✅ ACTIVÉ                         ║
║   • Format des données:           ✅ (n,14,2560)                    ║
║   • Channel names pour XAI:       ✅ SAUVEGARDÉS                    ║
║                                                                       ║
║   🚀 TU PEUX LANCER LE TRAITEMENT COMPLET EN TOUTE CONFIANCE !      ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
""")
elif confidence_score >= 5:
    print("""
╔═══════════════════════════════════════════════════════════════════════╗
║                                                                       ║
║   ⚠️⚠️⚠️ CONFIANCE PARTIELLE - VÉRIFIE LES POINTS ROUGES ⚠️⚠️⚠️    ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
""")
else:
    print("""
╔═══════════════════════════════════════════════════════════════════════╗
║                                                                       ║
║   ❌❌❌ PROBLÈME DÉTECTÉ - CORRIGE AVANT DE CONTINUER ❌❌❌       ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
""")

print("\n" + "="*80)
print("✅ VÉRIFICATION TERMINÉE")
print("="*80)